In [1]:
# %% [markdown]
# # 03_pilot_run.ipynb
# Pilot run of the v2 Semantic Layer architecture on a stratified
# 18-question sample (1 question per category x difficulty combo)
# x 3 iterations = 54 API calls.
#
# Purpose: verify before spending the full 600-call budget that:
#   1. The LLM reliably produces valid concept-SQL (low compile
#      error rate)
#   2. Execution accuracy is plausible
#   3. The 3 known HIGH-risk questions from coverage analysis
#      (CR01, CR10, CR34) are included and their outcome specifically
#      checked -- these test whether the WHERE-clause aggregate guard
#      in semantic_compiler.py (added after the previous pilot
#      attempt failed on a DB path issue before reaching this point)
#      actually prevents the measure-misuse failure mode in practice
#
# Requires: utils.py, semantic_compiler.py, financial_semantic_layer.yaml,
# credit_rating_questions_all.json, and a working DB_PATH (verified in
# 01_fetch_bird_data.ipynb) all in this directory, plus OPENAI_API_KEY
# set in .env.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import pandas as pd
import time
from utils import query_semantic_layer_v2, evaluate, DB_PATH

print(f"DB_PATH in use: {DB_PATH}")

with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    all_questions = json.load(f)

df_all = pd.DataFrame(all_questions)
print(f"Loaded {len(df_all)} total questions.")

# %% [markdown]
# ## 1. Build the stratified pilot sample
#     1 question per (category, difficulty) combination where
#     available, prioritizing inclusion of the 3 known HIGH-risk
#     questions from coverage analysis.

# %%
KNOWN_HIGH_RISK_IDS = {'CR01', 'CR10', 'CR34'}

sample_rows = []
for category in df_all['category'].unique():
    for difficulty in ['simple', 'moderate', 'challenging']:
        subset = df_all[(df_all['category'] == category) & (df_all['difficulty'] == difficulty)]
        if len(subset) == 0:
            continue
        # prefer a known HIGH-risk question if one exists in this
        # stratum, so the pilot specifically exercises the WHERE-guard
        risk_match = subset[subset['question_id'].isin(KNOWN_HIGH_RISK_IDS)]
        chosen = risk_match.iloc[0] if len(risk_match) > 0 else subset.iloc[0]
        sample_rows.append(chosen)

df_pilot = pd.DataFrame(sample_rows).reset_index(drop=True)

included_high_risk = set(df_pilot['question_id']) & KNOWN_HIGH_RISK_IDS
print(f"Pilot sample: {len(df_pilot)} questions")
print(df_pilot[['question_id', 'difficulty', 'category']].to_string(index=False))
print(f"\nKnown HIGH-risk questions included: {sorted(included_high_risk)} "
      f"({len(included_high_risk)} / {len(KNOWN_HIGH_RISK_IDS)})")
if len(included_high_risk) < len(KNOWN_HIGH_RISK_IDS):
    missing = KNOWN_HIGH_RISK_IDS - included_high_risk
    print(f"  NOTE: {missing} not included (a different question occupies")
    print(f"  that category/difficulty stratum). Not a problem -- these will")
    print(f"  still be covered in the full 60-question run.")

N_ITER_PILOT = 3
print(f"\nTotal API calls: {len(df_pilot)} questions x {N_ITER_PILOT} iterations = {len(df_pilot) * N_ITER_PILOT}")

# %% [markdown]
# ## 2. Run the pilot

# %%
PILOT_RESULTS_PATH = './results_pilot_v2.csv'

results = []
print("\n" + "=" * 70)
print("RUNNING PILOT")
print("=" * 70)

for i, row in df_pilot.iterrows():
    q_id = row['question_id']
    question = row['question']
    evidence = row['evidence']
    gold_sql = row['SQL']
    difficulty = row['difficulty']
    category = row['category']

    for iteration in range(N_ITER_PILOT):
        try:
            res = query_semantic_layer_v2(question, evidence)
        except Exception as e:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'is_correct': False,
                'compile_error': None, 'execution_error': f"API call failed: {e}",
                'concept_sql': None, 'compiled_sql': None, 'gold_sql': gold_sql,
                'latency_ms': None, 'prompt_chars': None, 'substitutions': None,
            })
            print(f"  [{q_id} iter{iteration}] API ERROR: {e}")
            continue

        if res['compile_error']:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'is_correct': False,
                'compile_error': res['compile_error'], 'execution_error': None,
                'concept_sql': res['concept_sql'], 'compiled_sql': None, 'gold_sql': gold_sql,
                'latency_ms': res['latency_ms'], 'prompt_chars': res['prompt_chars'],
                'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
            })
            flag = " <-- KNOWN HIGH-RISK QUESTION" if q_id in KNOWN_HIGH_RISK_IDS else ""
            print(f"  [{q_id} iter{iteration}] COMPILE ERROR: {res['compile_error'][:80]}{flag}")
            continue

        ev = evaluate(res['sql'], gold_sql)
        correct = bool(ev['is_correct']) if ev['is_correct'] is not None else False

        results.append({
            'question_id': q_id, 'difficulty': difficulty, 'category': category,
            'iteration': iteration, 'is_correct': correct,
            'compile_error': None, 'execution_error': ev.get('error'),
            'concept_sql': res['concept_sql'], 'compiled_sql': res['sql'], 'gold_sql': gold_sql,
            'latency_ms': res['latency_ms'], 'prompt_chars': res['prompt_chars'],
            'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
        })
        status = "correct" if correct else ("EXEC ERROR" if ev.get('error') else "wrong result")
        flag = " <-- KNOWN HIGH-RISK QUESTION" if q_id in KNOWN_HIGH_RISK_IDS else ""
        print(f"  [{q_id} iter{iteration}] {status}{flag}")

        time.sleep(0.3)

    pd.DataFrame(results).to_csv(PILOT_RESULTS_PATH, index=False, encoding='utf-8-sig')

print("\n" + "=" * 70)
print(f"Pilot complete. Saved: {PILOT_RESULTS_PATH}")
print("=" * 70)

# %% [markdown]
# ## 3. Overall pilot analysis

# %%
df_res = pd.DataFrame(results)

print(f"Total pilot records: {len(df_res)}")
print(f"Overall accuracy: {df_res['is_correct'].mean():.1%}")

n_compile_err = df_res['compile_error'].notna().sum()
n_exec_err = df_res['execution_error'].notna().sum()
n_correct = df_res['is_correct'].sum()
n_wrong_result = len(df_res) - n_compile_err - n_exec_err - n_correct

print(f"\n--- Two-stage failure taxonomy ---")
print(f"  Correct                                  : {n_correct} ({n_correct/len(df_res):.1%})")
print(f"  Compile errors (concept-SQL invalid)      : {n_compile_err} ({n_compile_err/len(df_res):.1%})")
print(f"  Execution errors (compiled SQL invalid)   : {n_exec_err} ({n_exec_err/len(df_res):.1%})")
print(f"  Wrong result (ran fine, answer mismatched): {n_wrong_result} ({n_wrong_result/len(df_res):.1%})")

# %% [markdown]
# ## 4. Specific check: did the WHERE-guard fix hold up on the 3
#     known HIGH-risk questions?

# %%
print("=" * 70)
print("HIGH-RISK QUESTION OUTCOME CHECK (CR01, CR10, CR34)")
print("=" * 70)

df_risk = df_res[df_res['question_id'].isin(KNOWN_HIGH_RISK_IDS)]

if len(df_risk) == 0:
    print("None of the 3 known HIGH-risk questions were in this pilot sample")
    print("(a different question occupied their category/difficulty stratum).")
    print("They will be covered in the full 60-question run instead.")
else:
    for qid in sorted(df_risk['question_id'].unique()):
        sub = df_risk[df_risk['question_id'] == qid]
        print(f"\n[{qid}] {len(sub)} iterations:")
        for _, r in sub.iterrows():
            if r['compile_error']:
                is_where_guard = 'WHERE clause' in str(r['compile_error'])
                guard_note = " (caught by WHERE-guard -- this is the fix working as intended)" if is_where_guard else ""
                print(f"  iter{r['iteration']}: COMPILE ERROR{guard_note}")
                print(f"    concept_sql: {r['concept_sql']}")
                print(f"    error: {r['compile_error'][:150]}")
            elif r['execution_error']:
                print(f"  iter{r['iteration']}: EXECUTION ERROR: {r['execution_error'][:100]}")
            elif r['is_correct']:
                print(f"  iter{r['iteration']}: ✓ correct")
            else:
                print(f"  iter{r['iteration']}: wrong result")
                print(f"    concept_sql: {r['concept_sql']}")
                print(f"    compiled_sql: {r['compiled_sql']}")

    print("""

INTERPRETATION:
- If these show "COMPILE ERROR (caught by WHERE-guard...)": the LLM
  attempted the same measure-misuse pattern as before, but the
  compiler correctly rejected it with a clean, informative error
  instead of producing invalid SQL. This is a SUCCESSFUL defense,
  even though it still counts as a wrong answer for accuracy purposes.
- If these show "✓ correct": the added few-shot example successfully
  steered the LLM away from the misuse pattern entirely -- the
  strongest possible outcome.
- If these show "EXECUTION ERROR" with SQL that doesn't look like the
  WHERE-guard pattern: a NEW failure mode has appeared. Inspect the
  concept_sql/compiled_sql above to diagnose.
""")

# %% [markdown]
# ## 5. Full compile-error detail (for any remaining prompt refinement)

# %%
if n_compile_err > 0:
    print("=" * 70)
    print("ALL COMPILE ERRORS")
    print("=" * 70)
    for _, row in df_res[df_res['compile_error'].notna()].iterrows():
        flag = " [KNOWN HIGH-RISK]" if row['question_id'] in KNOWN_HIGH_RISK_IDS else ""
        print(f"\n[{row['question_id']} iter{row['iteration']}]{flag}")
        print(f"  Concept-SQL: {row['concept_sql']}")
        print(f"  Error: {row['compile_error']}")
else:
    print("No compile errors in this pilot -- clean run.")

# %% [markdown]
# ## 6. Full execution-error detail

# %%
if n_exec_err > 0:
    print("=" * 70)
    print("ALL EXECUTION ERRORS")
    print("=" * 70)
    for _, row in df_res[df_res['execution_error'].notna()].iterrows():
        print(f"\n[{row['question_id']} iter{row['iteration']}]")
        print(f"  Concept-SQL: {row['concept_sql']}")
        print(f"  Compiled SQL: {row['compiled_sql']}")
        print(f"  Error: {row['execution_error']}")
else:
    print("No execution errors in this pilot -- clean run.")

# %% [markdown]
# ## 7. Accuracy breakdown

# %%
print("=" * 70)
print("ACCURACY BY DIFFICULTY")
print("=" * 70)
print(df_res.groupby('difficulty')['is_correct'].agg(['mean', 'count']))

print("\n" + "=" * 70)
print("ACCURACY BY CATEGORY")
print("=" * 70)
print(df_res.groupby('category')['is_correct'].agg(['mean', 'count']))

print("""

NEXT STEPS:
  - If compile error rate is low (<15%) and the HIGH-risk questions
    show the WHERE-guard working (or the few-shot preventing misuse
    entirely), proceed to the full 60-question x 5-iteration run.
  - If compile error rate is still high, or a NEW failure pattern
    appears in section 4/5 above, that needs a fix in
    semantic_compiler.py or utils.py's few-shot examples BEFORE the
    full run -- re-run this pilot after any fix to confirm before
    spending the full budget.
""")

SCRIPT VERSION: 2026-07-15-v1
DB_PATH in use: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Loaded 60 total questions.
Pilot sample: 18 questions
question_id  difficulty             category
       CR01      simple         default_risk
       CR11    moderate         default_risk
       CR21 challenging         default_risk
       CR04      simple       loan_portfolio
       CR17    moderate       loan_portfolio
       CR25 challenging       loan_portfolio
       CR06      simple       client_profile
       CR18    moderate       client_profile
       CR26 challenging       client_profile
       CR34      simple        regional_risk
       CR13    moderate        regional_risk
       CR24 challenging        regional_risk
       CR08      simple transaction_behavior
       CR15    moderate transaction_behavior
       CR23 challenging transaction_behavior
       CR10      simple            card_risk
       CR19    moderate            card_risk
       CR27 chal

In [2]:
# %% [markdown]
# # 03b_pilot_diagnostic.ipynb
# Loads the already-saved results_pilot_v2.csv and extracts exactly
# the sections needed to diagnose:
#   1. The compile/execution/wrong-result breakdown (taxonomy)
#   2. The 3 known HIGH-risk question outcomes (CR01, CR10, CR34)
#   3. Why client_profile and transaction_behavior scored 0%
#
# No API calls -- purely reads the CSV already on disk from the
# pilot run.

# %%
import pandas as pd
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

df = pd.read_csv('results_pilot_v2.csv')
print(f"Loaded {len(df)} rows from results_pilot_v2.csv")

KNOWN_HIGH_RISK_IDS = {'CR01', 'CR10', 'CR34'}
ZERO_ACCURACY_CATEGORIES = {'client_profile', 'transaction_behavior'}

# %% [markdown]
# ## 1. Failure taxonomy (compile vs execution vs wrong-result vs correct)

# %%
n_total = len(df)
n_compile_err = df['compile_error'].notna().sum()
n_exec_err = df['execution_error'].notna().sum()
n_correct = df['is_correct'].sum()
n_wrong_result = n_total - n_compile_err - n_exec_err - n_correct

print("=" * 70)
print("OVERALL TAXONOMY")
print("=" * 70)
print(f"  Correct        : {n_correct} ({n_correct/n_total:.1%})")
print(f"  Compile errors : {n_compile_err} ({n_compile_err/n_total:.1%})")
print(f"  Execution errors: {n_exec_err} ({n_exec_err/n_total:.1%})")
print(f"  Wrong result   : {n_wrong_result} ({n_wrong_result/n_total:.1%})")

# %% [markdown]
# ## 2. Taxonomy broken down BY CATEGORY -- this is the key diagnostic
#     for why client_profile / transaction_behavior scored 0%

# %%
def taxonomy_row(g):
    n = len(g)
    return pd.Series({
        'n': n,
        'correct': g['is_correct'].sum(),
        'compile_err': g['compile_error'].notna().sum(),
        'exec_err': g['execution_error'].notna().sum(),
        'wrong_result': n - g['is_correct'].sum() - g['compile_error'].notna().sum() - g['execution_error'].notna().sum(),
    })

print("=" * 70)
print("TAXONOMY BY CATEGORY")
print("=" * 70)
try:
    by_cat = df.groupby('category').apply(taxonomy_row, include_groups=False)
except TypeError:
    # include_groups was only added in pandas 2.2+; fall back for
    # older installs (harmless -- taxonomy_row doesn't use the
    # grouping column itself, so the extra column pandas may include
    # in older versions doesn't affect the computed values)
    by_cat = df.groupby('category').apply(taxonomy_row)
print(by_cat)

# %% [markdown]
# ## 3. Full detail for the two 0%-accuracy categories: every row's
#     concept_sql, compiled_sql, and whichever error applies

# %%
print("=" * 70)
print("FULL DETAIL: client_profile and transaction_behavior")
print("=" * 70)

df_zero = df[df['category'].isin(ZERO_ACCURACY_CATEGORIES)]

for _, row in df_zero.iterrows():
    print(f"\n--- [{row['question_id']}] iter{row['iteration']} ({row['category']}, {row['difficulty']}) ---")
    print(f"  concept_sql : {row['concept_sql']}")
    if pd.notna(row['compile_error']):
        print(f"  COMPILE ERROR: {row['compile_error']}")
    elif pd.notna(row['execution_error']):
        print(f"  compiled_sql: {row['compiled_sql']}")
        print(f"  EXECUTION ERROR: {row['execution_error']}")
    elif not row['is_correct']:
        print(f"  compiled_sql: {row['compiled_sql']}")
        print(f"  gold_sql    : {row['gold_sql']}")
        print(f"  -> ran successfully but WRONG RESULT")
    else:
        print(f"  -> correct (unexpected given 0% category average -- check for a mixed result)")

# %% [markdown]
# ## 4. HIGH-risk question outcomes (CR01, CR10, CR34)

# %%
print("=" * 70)
print("HIGH-RISK QUESTION DETAIL: CR01, CR10, CR34")
print("=" * 70)

df_risk = df[df['question_id'].isin(KNOWN_HIGH_RISK_IDS)]

for qid in sorted(df_risk['question_id'].unique()):
    sub = df_risk[df_risk['question_id'] == qid]
    print(f"\n[{qid}] {len(sub)} iterations:")
    for _, r in sub.iterrows():
        if pd.notna(r['compile_error']):
            is_where_guard = 'WHERE clause' in str(r['compile_error'])
            note = " <-- WHERE-guard caught it (fix working)" if is_where_guard else " <-- different compile error"
            print(f"  iter{r['iteration']}: COMPILE ERROR{note}")
            print(f"    concept_sql: {r['concept_sql']}")
            print(f"    error: {r['compile_error']}")
        elif pd.notna(r['execution_error']):
            print(f"  iter{r['iteration']}: EXECUTION ERROR: {r['execution_error']}")
            print(f"    compiled_sql: {r['compiled_sql']}")
        elif r['is_correct']:
            print(f"  iter{r['iteration']}: ✓ correct")
        else:
            print(f"  iter{r['iteration']}: wrong result")
            print(f"    compiled_sql: {r['compiled_sql']}")
            print(f"    gold_sql    : {r['gold_sql']}")

# %% [markdown]
# ## 5. Quick pattern scan: are the client_profile/transaction_behavior
#     failures clustered on ONE question (an isolated bug) or spread
#     across ALL questions in those categories (a systemic prompt/
#     concept issue)?

# %%
print("=" * 70)
print("FAILURE CLUSTERING CHECK")
print("=" * 70)
for cat in ZERO_ACCURACY_CATEGORIES:
    sub = df[df['category'] == cat]
    qids = sub['question_id'].unique()
    print(f"\n{cat}: {len(qids)} distinct question(s) in pilot sample: {sorted(qids)}")
    for qid in sorted(qids):
        qsub = sub[sub['question_id'] == qid]
        n_compile = qsub['compile_error'].notna().sum()
        n_exec = qsub['execution_error'].notna().sum()
        n_wrong = len(qsub) - n_compile - n_exec - qsub['is_correct'].sum()
        print(f"  [{qid}]: {len(qsub)} iters -> compile_err={n_compile}, exec_err={n_exec}, wrong_result={n_wrong}, correct={qsub['is_correct'].sum()}")

print("""

INTERPRETATION GUIDE:
- If ALL iterations of ALL questions in a 0%-category show
  compile_error: the concept vocabulary or few-shot examples for that
  category's query pattern need work -- the LLM can't produce valid
  concept-SQL for this category at all yet.
- If compile succeeds but execution_error or wrong_result dominates:
  the concept-SQL is syntactically fine but semantically off -- check
  whether a specific concept (e.g. birth_year, a join pattern) is
  being used incorrectly, or whether the compiled SQL structurally
  diverges from what gold_sql expects.
- If only ONE question_id in a category is failing across all 3
  categories represented, but the pilot only sampled 1-2 questions
  per category, this may not generalize -- the full 60-question run
  will show whether it's isolated or category-wide.
""")

Loaded 54 rows from results_pilot_v2.csv
OVERALL TAXONOMY
  Correct        : 15 (27.8%)
  Compile errors : 19 (35.2%)
  Execution errors: 7 (13.0%)
  Wrong result   : 13 (24.1%)
TAXONOMY BY CATEGORY
                      n  correct  compile_err  exec_err  wrong_result
category                                                             
card_risk             9        3            4         0             2
client_profile        9        1            3         0             5
default_risk          9        3            6         0             0
loan_portfolio        9        6            0         3             0
regional_risk         9        0            5         3             1
transaction_behavior  9        2            1         1             5
FULL DETAIL: client_profile and transaction_behavior

--- [CR06] iter0 (client_profile, simple) ---
  concept_sql : SELECT client.gender, COUNT(DISTINCT client.client_id)
   FROM client
   JOIN disp ON disp.client_link
   JOIN loan ON disp.a